# E3 seed verification (P3-T5 and P0-T2 acceptance)

Two acceptance criteria are still open on the case study, and both are
reproducibility claims that were asserted but never tested:

1. **P3-T5** requires that three randomly chosen seeds be re-run and
   reproduce their stored rows *exactly*. The confirmatory shards ran across
   ten different Colab hosts, so this doubles as a cross-machine check.
2. **P0-T2** requires that a replication give bit-identical metrics under
   different worker counts and different shard boundaries.

`CALIBRATION.md` §3.6 also asks which model implementations honour an
explicit seed. **That needs no separate experiment**: every metric column is
namespaced by model, so a re-run that reproduces a model's columns proves
that model honoured its seed, and any offender is named by its prefix.

Runtime is about 50 minutes: 3 replications at n = 500 for Test A and 8 at
n = 100 for Test B. One Colab session, comfortably inside the cap.

**A failure here is important and must not be worked around.** If a model
does not honour its seed, the case-study data is not reproducible and the
Data Availability statement cannot claim that it is.

In [ ]:
import os
import sys
import subprocess

import numpy as np
import pandas as pd

REPO_URL = "https://github.com/hugogobato/Test-Informed-Simulation-Count-Algorithm-TISCA.git"
CANDIDATES = [
    os.path.abspath(os.path.join(os.getcwd(), "..")),          # notebooks/ in a checkout
    os.getcwd(),
    "/content/Test-Informed-Simulation-Count-Algorithm-TISCA",
    "/content/TISCA_repo",
]
REPO_ROOT = next((p for p in CANDIDATES
                  if os.path.isdir(os.path.join(p, "tisca", "python"))), None)
if REPO_ROOT is None:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "/content/TISCA_repo"], check=True)
    REPO_ROOT = "/content/TISCA_repo"
sys.path.insert(0, os.path.join(REPO_ROOT, "tisca", "python"))

RESULTS = os.path.join(REPO_ROOT, "results")
FIGURES = os.path.join(REPO_ROOT, "figures")
os.makedirs(FIGURES, exist_ok=True)
print("repo:", REPO_ROOT)


def download(path):
    """Colab download fallback (standing rule); a no-op off Colab."""
    try:
        from google.colab import files
        files.download(path)
        print("Downloaded:", path)
    except Exception as e:
        print("(Not on Colab / download skipped):", e)

In [ ]:
import os, platform, subprocess, time

def sh(cmd):
    p = subprocess.run(
        cmd, shell=True, capture_output=True, text=True,
        encoding="utf-8", errors="replace")
    return p.stdout.strip()

print("hostname:", platform.node() or "n/a")
print("os:", platform.platform())
print("nproc:", sh("nproc"))
print("cpu:", sh("grep -m1 -E 'model name' /proc/cpuinfo") or "n/a")
print(sh("free -g") or "RAM information unavailable")
print("mc.cores is fixed at 2; model fits are fixed to one thread.")


In [ ]:
import subprocess
p = subprocess.run(
    ["bash", "-lc", "apt-get -qq update >/dev/null && "
     "apt-get -qq install -y --no-install-recommends "
     "r-base r-base-dev libcurl4-openssl-dev >/dev/null 2>&1"],
    capture_output=True, text=True, encoding="utf-8", errors="replace")
if p.returncode != 0:
    print(p.stdout[-1000:])
    print(p.stderr[-2000:])
    raise RuntimeError("R installation failed")
print(subprocess.check_output(
    ["R", "--version"], text=True,
    encoding="utf-8", errors="replace").splitlines()[0])


In [ ]:
import hashlib, os, re, shutil, subprocess, sys
from pathlib import Path

BUNDLE_FOLDER_URL = 'https://drive.google.com/drive/folders/1w3quuskj25CBOFCGG0mTRGUHcufPpdb3?usp=sharing'
BUNDLE_SHA256 = '12d223bc0fcef624c1ff4cc35c5d7ecc1b1f9b05aa84ecd9d9e4a5a3382bae3c'
assert re.fullmatch(r"[0-9a-fA-F]{64}", BUNDLE_SHA256), \
    "Bundle SHA256 is missing or malformed; regenerate the notebook."

BUNDLE_DOWNLOAD_DIR = Path("/content/tisca_bundle_download")
if BUNDLE_DOWNLOAD_DIR.exists():
    shutil.rmtree(BUNDLE_DOWNLOAD_DIR)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "gdown"],
    check=True,
)
download = subprocess.run(
    [sys.executable, "-m", "gdown", "--folder", BUNDLE_FOLDER_URL,
     "--output", str(BUNDLE_DOWNLOAD_DIR), "--remaining-ok"],
    capture_output=True, text=True,
    encoding="utf-8", errors="replace",
)
print(download.stdout[-4000:])
if download.returncode != 0:
    print(download.stderr[-4000:])
    raise RuntimeError("Google Drive bundle download failed")
tar_candidates = sorted(BUNDLE_DOWNLOAD_DIR.rglob("tisca_rlib.tar.gz"))
sha_candidates = sorted(BUNDLE_DOWNLOAD_DIR.rglob("tisca_rlib.sha256"))
assert len(tar_candidates) == 1, f"expected one tarball, found {tar_candidates}"
assert len(sha_candidates) <= 1, f"expected at most one checksum file, found {sha_candidates}"
if sha_candidates:
    published_sha = sha_candidates[0].read_text().split()[0].lower()
    assert published_sha == BUNDLE_SHA256.lower(), \
        "published tisca_rlib.sha256 differs from the generated notebook"
    print("verified published checksum sidecar:", sha_candidates[0])
else:
    print("no tisca_rlib.sha256 sidecar in the shared folder; "
          "verifying the tarball against the embedded SHA256")
download_path = "/content/_dl_tisca_rlib.tar.gz"
shutil.copy2(tar_candidates[0], download_path)
with open(download_path, "rb") as f:
    observed_sha = hashlib.sha256(f.read()).hexdigest()
assert observed_sha == BUNDLE_SHA256.lower(), "R library bundle SHA mismatch"
if os.path.isdir("/content/tisca_rlib"):
    shutil.rmtree("/content/tisca_rlib")
subprocess.run(["tar", "xzf", download_path, "-C", "/content"], check=True)
LIBDIR = "/content/tisca_rlib/rlib"
assert os.path.isdir(LIBDIR), "bundle did not restore the expected rlib directory"
print("bundle restored:", LIBDIR, "from", BUNDLE_FOLDER_URL)


In [ ]:
import os, urllib.request

RUNCELL_URL = (
    "https://raw.githubusercontent.com/hugogobato/"
    "Test-Informed-Simulation-Count-Algorithm-TISCA/main/"
    "experiments/E3_mvbcf_casestudy/run_cell.R"
)
MVBCF_CPP_URL = (
    "https://raw.githubusercontent.com/Nathan-McJames/MVBCF_Paper/"
    "main/MVBCF_Code.cpp"
)
os.makedirs("/content/e3", exist_ok=True)
urllib.request.urlretrieve(RUNCELL_URL, "/content/e3/run_cell.R")
# The upstream C++ is downloaded at runtime and is never committed here.
urllib.request.urlretrieve(MVBCF_CPP_URL, "/content/e3/MVBCF_Code.cpp")
assert os.path.getsize("/content/e3/run_cell.R") > 1000
assert os.path.getsize("/content/e3/MVBCF_Code.cpp") > 10000
with open("/content/e3/run_cell.R") as f:
    run_cell_source = f.read()
required_fixes = [
    "nthread = nthread_global",
    "num_threads = nthread_global",
    "acquired <- dir.create(lk",
    "num_gfr = 0",
    "sigma2_leaf_init = 1^2 / n_tree_mu",
    "sigma2_leaf_init = 0.375^2 / n_tree_tau",
    'propensity_covariate = "prognostic"',
    "sample_sigma2_leaf = FALSE",
]
missing_fixes = [item for item in required_fixes if item not in run_cell_source]
assert not missing_fixes, (
    "GitHub main is serving a stale run_cell.R. Commit and push the "
    f"corrected driver before running this notebook; missing: {missing_fixes}"
)
print("downloaded run_cell.R and upstream MVBCF_Code.cpp")


In [ ]:
import os, subprocess

compile_script = "\n".join([
    ".libPaths(c('/content/tisca_rlib/rlib', .libPaths()))",
    "if (!requireNamespace('Rcpp', quietly=TRUE) ||",
    "    !requireNamespace('RcppArmadillo', quietly=TRUE) ||",
    "    !requireNamespace('RcppDist', quietly=TRUE)) stop('bundle missing Rcpp dependencies')",
    "library(Rcpp)",
    "sourceCpp('/content/e3/MVBCF_Code.cpp')",
    "stopifnot(is.function(fast_bart))",
    "cat('FAST_BART_OK\\n')",
])
with open("/content/e3/compile.R", "w") as f:
    f.write(compile_script)
p = subprocess.run(["Rscript", "/content/e3/compile.R"],
                   capture_output=True, text=True,
                   encoding="utf-8", errors="replace")
print(p.stdout[-3000:])
if p.returncode != 0 or "FAST_BART_OK" not in p.stdout:
    print(p.stderr[-3000:])
    raise RuntimeError("upstream MVBCF C++ compilation failed")
print("fast_bart() compiled")


In [ ]:
import time

import glob
import os
import subprocess

import numpy as np
import pandas as pd

# --- what to verify -------------------------------------------------------- #
# Test A: exact reproduction of stored rows. n = 500 costs ~9.6 min/replication at
# one core, so three seeds is ~29 minutes -- which is exactly what the acceptance
# criterion asks for and no more.
TEST_A_DGP, TEST_A_N, TEST_A_SEEDS = 1, 500, None      # None -> drawn below
N_TEST_A_SEEDS = 3
TEST_A_RNG_SEED = 20260806                             # fixed: the choice is auditable

# Test B: the four-way identity test. Run at n = 100 (~2.7 min/replication) because
# it needs 4 configurations x 2 seeds = 8 replications, and the property under test
# (stream construction) does not depend on the training size.
TEST_B_DGP, TEST_B_N = 1, 100
TEST_B_SEEDS = [37, 38]        # contiguous, so a shard-offset run is well defined

WORK = "/content/e3_verify"
os.makedirs(WORK, exist_ok=True)


def find_stored(dgp, n, mode="confirmatory"):
    """Locate the committed shard CSVs for one cell."""
    pats = [
        os.path.join(REPO_ROOT, "results", "E3", f"DGP{dgp}_n{n}_{mode}_replications.csv"),
        os.path.join(REPO_ROOT, "notebooks", "E3_shards",
                     f"E3_DGP{dgp}_n{n}_{mode}_shard*.csv"),
        f"/content/E3_DGP{dgp}_n{n}_{mode}_shard*.csv",
        f"/content/*DGP{dgp}_n{n}*.csv",
    ]
    files = []
    for p in pats:
        files = sorted(glob.glob(p))
        if files:
            break
    if not files:
        raise FileNotFoundError(
            f"No stored rows for DGP{dgp} n={n}. Either commit the shard CSVs to "
            "notebooks/E3_shards/, or upload them to /content before running this "
            "notebook.")
    df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
    print(f"stored rows for DGP{dgp} n={n}: {len(df)} from {len(files)} file(s)")
    return df


STORED_A = find_stored(TEST_A_DGP, TEST_A_N)
rng = np.random.default_rng(TEST_A_RNG_SEED)
TEST_A_SEEDS = sorted(rng.choice(np.sort(STORED_A["seed"].unique()),
                                 size=N_TEST_A_SEEDS, replace=False).tolist())
print("seeds selected for exact re-run:", TEST_A_SEEDS)
print("hostnames that produced the stored rows:",
      sorted(STORED_A["hostname"].astype(str).unique()))

## Runner and comparison helpers

In [ ]:
def run_seeds(dgp, n, s_start, s_end, out_name, cores=1, mode="confirmatory"):
    """Invoke run_cell.R over one contiguous seed range and return the rows."""
    out = os.path.join(WORK, out_name)
    if os.path.exists(out):
        os.remove(out)
    cmd = ["Rscript", "/content/e3/run_cell.R", str(dgp), str(n),
           str(s_start), str(s_end), "--out", out,
           "--cores", str(cores), "--mode", mode]
    env = dict(os.environ, TISCA_MVBCF_CPP="/content/e3/MVBCF_Code.cpp")
    t0 = time.time()
    p = subprocess.run(cmd, capture_output=True, text=True,
                       encoding="utf-8", errors="replace", env=env)
    print(f"  {' '.join(cmd[2:])} -> {time.time() - t0:.0f}s")
    if p.returncode != 0:
        print(p.stdout[-3000:])
        print(p.stderr[-3000:])
        raise RuntimeError("run_cell.R failed")
    return pd.read_csv(out)


# Columns that MUST differ or are allowed to differ between runs: they record the
# execution environment, not the replication's result.
ENV_COLS = {"hostname", "git_sha", "session_hash", "replication_seconds",
            "fit_seconds_mvbcf", "fit_seconds_bcf1", "fit_seconds_bcf2",
            "fit_seconds_bart1", "fit_seconds_bart2", "fit_seconds_mvbart",
            "error_message"}


def compare_rows(a, b, label_a, label_b):
    """Exact comparison of every non-environment column, keyed on `seed`."""
    a = a.set_index("seed").sort_index()
    b = b.set_index("seed").sort_index()
    common = sorted(set(a.index) & set(b.index))
    assert common, f"no shared seeds between {label_a} and {label_b}"
    cols = [c for c in a.columns if c in b.columns and c not in ENV_COLS]
    rows = []
    for c in cols:
        x, y = a.loc[common, c], b.loc[common, c]
        if pd.api.types.is_numeric_dtype(x) and pd.api.types.is_numeric_dtype(y):
            both_nan = x.isna() & y.isna()
            d = (x - y).abs().where(~both_nan, 0.0)
            identical = bool(((d == 0) | both_nan).all())
            maxdiff = float(d.max())
        else:
            identical = bool((x.astype(str) == y.astype(str)).all())
            maxdiff = np.nan
        rows.append({"column": c, "identical": identical, "max_abs_diff": maxdiff,
                     "model": c.split("_")[0] if "_" in c else "meta"})
    out = pd.DataFrame(rows)
    n_bad = int((~out["identical"]).sum())
    print(f"{label_a} vs {label_b}: {len(cols)} columns compared over "
          f"{len(common)} seed(s); {n_bad} differ")
    if n_bad:
        print(out[~out["identical"]].sort_values("max_abs_diff", ascending=False)
              .head(20).to_string(index=False))
    return out

## Test A: exact reproduction of stored rows

Environment columns (`hostname`, `git_sha`, `session_hash`, and every timing
column) are excluded from the comparison because they record where the run
happened, not what it produced. Everything else must match exactly, not
approximately: a tolerance here would hide precisely the drift the check
exists to detect.

In [ ]:
# --------------------------------------------------------------------------- #
# Test A: do three randomly chosen seeds reproduce their stored rows exactly?   #
# --------------------------------------------------------------------------- #
frames = []
for s in TEST_A_SEEDS:
    frames.append(run_seeds(TEST_A_DGP, TEST_A_N, s, s, f"A_seed{s}.csv", cores=1))
rerun_A = pd.concat(frames, ignore_index=True)
cmp_A = compare_rows(STORED_A[STORED_A["seed"].isin(TEST_A_SEEDS)], rerun_A,
                     "stored", "re-run")

### Per-model verdict (CALIBRATION.md §3.6)

In [ ]:
# Per-model verdict. This is the answer to CALIBRATION.md section 3.6: a model that
# honours its seed reproduces all of its columns; one that does not shows up here
# under its own prefix, with no extra fitting required.
per_model = (cmp_A[cmp_A["model"].isin(["mvbcf", "bcf", "bart", "mvbart"])]
             .groupby("model")
             .agg(columns=("column", "size"),
                  n_differing=("identical", lambda v: int((~v).sum())),
                  max_abs_diff=("max_abs_diff", "max")))
per_model["honours_seed"] = per_model["n_differing"] == 0
print(per_model.to_string())
print()
meta = cmp_A[~cmp_A["model"].isin(["mvbcf", "bcf", "bart", "mvbart"])]
print("non-model columns differing:",
      meta.loc[~meta["identical"], "column"].tolist() or "none")

## Test B: the four-way identity test

In [ ]:
# --------------------------------------------------------------------------- #
# Test B: the four-way identity test (P0-T2 acceptance, CALIBRATION.md 3.5).     #
# --------------------------------------------------------------------------- #
# mc.cores in {1, 2} x {shard-aligned, shard-offset}. "Shard-offset" means the same
# seeds are requested as part of a range that starts earlier, which is exactly what
# happens when a cell is split into shards; the stream for index j must not depend
# on where its shard began.
lo, hi = min(TEST_B_SEEDS), max(TEST_B_SEEDS)
variants = {}
variants["cores1_aligned"] = run_seeds(TEST_B_DGP, TEST_B_N, lo, hi,
                                       "B_c1_aligned.csv", cores=1)
variants["cores2_aligned"] = run_seeds(TEST_B_DGP, TEST_B_N, lo, hi,
                                       "B_c2_aligned.csv", cores=2)
off = max(0, lo - 2)
variants["cores1_offset"] = run_seeds(TEST_B_DGP, TEST_B_N, off, hi,
                                      "B_c1_offset.csv", cores=1)
variants["cores2_offset"] = run_seeds(TEST_B_DGP, TEST_B_N, off, hi,
                                      "B_c2_offset.csv", cores=2)

ref = "cores1_aligned"
cmp_B = {}
for k, v in variants.items():
    if k == ref:
        continue
    cmp_B[k] = compare_rows(variants[ref], v, ref, k)

## Verdict, and the block to paste into CALIBRATION.md

In [ ]:
# --------------------------------------------------------------------------- #
# Verdict                                                                       #
# --------------------------------------------------------------------------- #
checks = []
checks.append({
    "check": "P3-T5: 3 random seeds reproduce their stored rows exactly",
    "detail": f"seeds {TEST_A_SEEDS}, {len(cmp_A)} columns each; "
              f"{int((~cmp_A['identical']).sum())} differing",
    "pass": bool(cmp_A["identical"].all()),
})
checks.append({
    "check": "cross-machine: the stored rows came from a different host",
    "detail": f"stored hosts {sorted(STORED_A['hostname'].astype(str).unique())} "
              f"vs this session",
    "pass": True,       # informational; the exactness check above is the real test
})
for k, c in cmp_B.items():
    checks.append({
        "check": f"P0-T2 four-way identity: {ref} == {k}",
        "detail": f"{int((~c['identical']).sum())} of {len(c)} columns differ",
        "pass": bool(c["identical"].all()),
    })
for m, r in per_model.iterrows():
    checks.append({
        "check": f"seed honoured by {m}",
        "detail": f"{int(r['n_differing'])} of {int(r['columns'])} columns differ, "
                  f"max |diff| = {r['max_abs_diff']}",
        "pass": bool(r["honours_seed"]),
    })
report = pd.DataFrame(checks)
print(report.to_string(index=False))
print()
print("ALL PASS" if report["pass"].all() else "*** FAILURES ABOVE -- do not tick "
      "the CALIBRATION.md boxes ***")

os.makedirs(os.path.join(RESULTS, "E3"), exist_ok=True)
report.to_csv(os.path.join(RESULTS, "E3", "seed_verification.csv"), index=False)
cmp_A.to_csv(os.path.join(RESULTS, "E3", "seed_verification_columns.csv"), index=False)
download(os.path.join(RESULTS, "E3", "seed_verification.csv"))

print()
print("--- paste into experiments/E3_mvbcf_casestudy/CALIBRATION.md ---")
print(f"""
### Seed verification, {pd.Timestamp.today().date()}

- [{'x' if report['pass'].all() else ' '}] P3-T5 acceptance: seeds {TEST_A_SEEDS} of
      DGP{TEST_A_DGP} n={TEST_A_N} re-run and compared against the stored rows on
      {len(cmp_A)} non-environment columns. Differing columns:
      {int((~cmp_A['identical']).sum())}.
- [{'x' if all(c['identical'].all() for c in cmp_B.values()) else ' '}] P0-T2 §3.5
      four-way identity test (mc.cores 1 vs 2, shard-aligned vs shard-offset) on
      DGP{TEST_B_DGP} n={TEST_B_N}, seeds {TEST_B_SEEDS}.
- [{'x' if bool(per_model['honours_seed'].all()) else ' '}] P0-T2 §3.6 per-model seed
      honouring, read off the column-level comparison:
      {dict(per_model['honours_seed'])}.
""")